# Step 3b — Graded extraction (v2)

**Settings: GPU T4 x2, Internet On.** Attach the competition data.

## What v1 taught us

Binary `present/absent/not_mentioned` threw away the signal. Medial OA scored **recall 0.07** on the binary field while its *severity* field ranked cleanly: none 0.06 → mild 0.57 → moderate 0.67 → severe 1.00.

The model could read the finding. The output format couldn't express it.

**And the competition metric is AUC, which scores ranking, not classification.** You never need a hard yes/no — you need the positives ordered above the negatives. So v2 asks for a 0–100 likelihood per finding and we score with AUC, the actual metric.

Cell 1 re-scores your existing v1 predictions with AUC (free, no GPU) so you can see the difference before spending anything.

In [ ]:
# --- CELL 1: re-score v1 predictions with AUC (no GPU needed) -------------
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

# graded map for the v1 categorical output
CAT = {"present": 1.0, "not_mentioned": 0.30, "absent": 0.0}
SEV = {"none": 0.0, "mild": 0.55, "moderate": 0.80, "severe": 1.0}

def v1_score(p, key):
    base = CAT.get(str(p.get(key, "not_mentioned")).lower(), 0.30)
    if key in ("medial_oa", "lateral_oa", "pf_oa"):
        s = SEV.get(str(p.get(f"{key}_severity", "none")).lower(), None)
        if s is not None:
            # severity dominates: it carried the real signal in v1
            return 0.25 * base + 0.75 * s
    return base

print(f"{'label':<18}{'AUC(v1 graded)':>16}{'n_pos':>7}")
aucs = []
for key, col in KEYMAP.items():
    y = pd.to_numeric(gold[col], errors="coerce").fillna(0).astype(int).values
    s = np.array([v1_score(p, key) for p in preds])
    if y.min() == y.max():
        print(f"{col:<18}{'n/a (one class)':>16}{y.sum():>7}"); continue
    a = roc_auc_score(y, s); aucs.append(a)
    print(f"{col:<18}{a:>16.3f}{y.sum():>7}")
print(f"\nMACRO AUC (v1, graded rescoring): {np.mean(aucs):.3f}")
print("note: n=58, so each label's AUC has a standard error near +/-0.08")

## v2 prompt — graded likelihood for every finding

Changes from v1:

1. **0–100 likelihood** per finding instead of a category. Preserves ordering, which is all AUC needs.
2. **No "not mentioned" escape hatch.** Silence is evidence, not an abstention — the model gives a low-but-nonzero score instead of refusing to answer.
3. **Describe the finding, don't name it.** v1 failed on OA because reports say "chondral thinning" rather than "osteoarthritis". v2 lists the synonyms explicitly.
4. **Calibration anchors** in the prompt so the numbers mean something consistent across reports.

In [ ]:
SYSTEM2 = """You are a musculoskeletal radiologist reading knee MRI reports and estimating how likely each of twelve findings is to be truly present.

The report may be in ANY language (English, Spanish, Portuguese, German, French, Italian, Turkish, Dutch and others). Read it in its original language. Do not translate.

For EACH finding give an integer 0-100: the likelihood that an expert radiologist reviewing the actual images would mark this finding as present.

Use this scale:
  0-10    explicitly negated ("ACL intact", "menisci normal", "no effusion")
  11-30   not mentioned at all, and the report is otherwise thorough
  31-55   equivocal, borderline, trace, minimal, or degenerative change short of a tear
  56-80   clearly described but mild, or described indirectly
  81-100  explicitly and unambiguously stated as present

THE TWELVE FINDINGS (match on MEANING, not exact wording -- reports rarely use these exact terms):

acl              anterior cruciate ligament tear, rupture, discontinuity
mcl              medial collateral ligament tear, rupture, high-grade sprain
medial_meniscus  medial meniscus TEAR (degenerative signal alone is NOT a tear -> low score)
lateral_meniscus lateral meniscus TEAR (same rule)
medial_oa        MEDIAL tibiofemoral compartment: osteoarthritis, cartilage/chondral loss or
                 thinning, joint space narrowing, subchondral sclerosis, osteophytes
lateral_oa       LATERAL tibiofemoral compartment: same features
pf_oa            PATELLOFEMORAL compartment (patella/trochlea): same features, chondromalacia
effusion         joint effusion, intra-articular fluid (trace/small still counts, score ~50-65)
synovitis        synovial thickening, synovial enhancement, inflammation, plica synovitis
bakers           Baker's cyst, popliteal cyst, gastrocnemio-semimembranosus bursa cyst
contusion        bone contusion, bone bruise, bone marrow oedema/signal from trauma
fracture         fracture, cortical break, avulsion, insufficiency or stress fracture

RULES:
- Negation is decisive. "Intact", "normal", "unremarkable", "no evidence of" => score under 10.
- Compartment matters. Only score medial_oa for the MEDIAL side, lateral_oa for the LATERAL side. A generic "tricompartmental osteoarthritis" raises ALL THREE.
- Meniscal degeneration/mucoid signal without a tear line => 20-40, not high.
- If a finding is never referred to anywhere, give 15-25. Never give exactly 0 for silence.

Reply with ONLY a JSON object of twelve integers, no prose:
{"acl":0,"mcl":0,"medial_meniscus":0,"lateral_meniscus":0,"medial_oa":0,"lateral_oa":0,"pf_oa":0,"effusion":0,"synovitis":0,"bakers":0,"contusion":0,"fracture":0}"""

def extract2(report, max_new_tokens=200):
    msgs = [{"role":"system","content":SYSTEM2},
            {"role":"user","content":f"REPORT:\n{report[:6000]}"}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids  = tok([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                             temperature=None, top_p=None, top_k=None,
                             pad_token_id=tok.eos_token_id)
    gen = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r"\{.*\}", gen, re.S)
    if not m: return None, gen
    try:    return json.loads(m.group(0)), gen
    except Exception: return None, gen

print("v2 ready")

In [ ]:
# --- run v2 over the 58 gold reports -------------------------------------
import time
preds2, fails2 = [], 0
t0 = time.time()
for i, r in gold.iterrows():
    p, raw = extract2(str(r["Report"]))
    if p is None: fails2 += 1
    preds2.append(p or {})
    if (i+1) % 10 == 0:
        print(f"  {i+1}/{len(gold)}  ({(time.time()-t0)/(i+1):.1f}s each)")
el = time.time()-t0
print(f"\ndone in {el/60:.1f} min | {el/len(gold):.1f}s per report | parse failures: {fails2}")
print(f"projected cost for all 4,349 unlabelled: {el/len(gold)*4349/3600:.1f} GPU-hours")

## Score v2 with AUC — the actual competition metric

Macro AUC across the twelve labels is exactly what the leaderboard computes. This number is directly comparable to the 0.809 public baseline, with the caveat that n=58 makes every estimate noisy (standard error roughly ±0.08 per label).

Read the **macro AUC** as your headline, and the per-label column to see which findings text alone cannot recover.

In [ ]:
def num(p, k):
    try:
        v = float(p.get(k, 20))
        return min(max(v, 0), 100)
    except Exception:
        return 20.0

print(f"{'label':<18}{'AUC v2':>9}{'AUC v1':>9}{'delta':>9}{'n_pos':>7}")
a2_all, a1_all = [], []
for key, col in KEYMAP.items():
    y = pd.to_numeric(gold[col], errors="coerce").fillna(0).astype(int).values
    if y.min() == y.max():
        print(f"{col:<18}{'n/a':>9}"); continue
    s2 = np.array([num(p, key) for p in preds2])
    s1 = np.array([v1_score(p, key) for p in preds])
    a2 = roc_auc_score(y, s2); a1 = roc_auc_score(y, s1)
    a2_all.append(a2); a1_all.append(a1)
    flag = "  <--" if a2 - a1 > 0.05 else ("  !!" if a2 - a1 < -0.05 else "")
    print(f"{col:<18}{a2:>9.3f}{a1:>9.3f}{a2-a1:>+9.3f}{y.sum():>7}{flag}")

print(f"\nMACRO AUC  v2 = {np.mean(a2_all):.3f}   v1 = {np.mean(a1_all):.3f}   delta = {np.mean(a2_all)-np.mean(a1_all):+.3f}")
print(f"public baseline to beat: 0.809")

pd.DataFrame(preds2).assign(StudyInstanceUID=gold.StudyInstanceUID).to_csv("gold58_extraction_v2.csv", index=False)
print("\nwrote gold58_extraction_v2.csv")

In [ ]:
# --- score distribution sanity check --------------------------------------
# If the model is anchoring on a few round numbers, or never uses the middle
# of the range, the ranking collapses and AUC suffers.
d = pd.DataFrame([{k: num(p, k) for k in KEYMAP} for p in preds2])
print("score distribution per label:")
print(d.describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]].round(1).to_string())
print("\nmost common values overall:")
print(pd.Series(d.values.ravel()).value_counts().head(12).to_string())